# Notebook 0: Prepare Grouped Dataset For Training

Notebook 2 egitiminden once kopya farkindalikli, source-aware ve taksonomi uyumlu dataset hazirlama akisidir.

Repo icinde hazir class-root datasetler: data/class_root_dataset/tomato_fruit, data/class_root_dataset/grape_fruit ve data/class_root_dataset/grape_leaf.

Audit hedefi sadece "temiz" veri degil, adapter performansi icin guvenilir veri uretmek olmalidir:
- exact duplicate ve near-duplicate ailelerini birlikte tutun.
- ayni capture/session/source ailelerini ayirmayin; source leakage'i engelleyin.
- gercek hard negatives ve OOD pool'u klasification siniflarindan ayri yonetin.
- sadece audit raporu temiz oldugu icin degil, class support ve source dagilimi guvenilir oldugu icin materyalize edin.

Onerilen kullanim sirasi:
1. Parametreleri duzenleyin.
2. Guncelleme/erisim kontrolu hucresini calistirin.
3. Audit hucresini calistirin.
4. Cikti dosyalarini `guided/00_start_here.md` ile okuyun.
5. Her sey temizse runtime dataset materyalizasyonunu acin.


## Hizli Akis

Bu notebookun amaci class-root datasetleri once audit etmek, sonra Notebook 2'nin tuketecegi `continual/`, `val/`, `test/`, `ood/`, `oe/` runtime yapisina hazirlamaktir.

- Genelde sadece `REPO_DATASET_NAME`, `CROP_NAME`, `PART_NAME` ve gerekiyorsa `OOD_*` alanlarini degistirin.
- Drive importu opsiyoneldir; repo icindeki `data/class_root_dataset` varsayilan yoldur.
- Audit raporunu okumadan materyalizasyonu kesin sonuc gibi kullanmayin. `guided/00_start_here.md` ilk bakilacak dosyadir.
- Runtime cikti hedefi: `data/prepared_runtime_datasets/<crop>__<part>/`.


## Repo-Ici Dataset Secimi

Notebook 0, repo icindeki class-root datasetleri audit edip runtime dataset'e cevirir. Asagidaki adimlari izleyin:

### Nasil Kullanilir

1. `REPO_DATASET_ROOT` ile datasetlerin bulundugu repo klasorunu secin.
2. `REPO_DATASET_NAME` ile alt dataset adini belirtin ya da bos birakip notebook'un sormasini bekleyin.
3. `DATASET_ROOT` bos ise yukaridaki secim kullanilir.
4. Audit ve materyalizasyon hucrelerini sirayla calistirin.

### Ornek

```python
REPO_DATASET_ROOT = "data/class_root_dataset"
REPO_DATASET_NAME = "tomato_fruit"
```

Bu akiste veri importu Drive'a bagimli degildir; Notebook 0 repo icindeki veri kaynaklarini audit edip runtime dataset'e cevirir.

## GPU Bellek Optimizasyonu (T4 Colab)

Notebook 0, Colab T4 GPU'da kalitiyle çalışması için otomatik bellek adaptasyonunu destekler:

- **T4 GPU** (~16GB): embedding batch size = 8
- **V100 GPU** (~32GB): embedding batch size = 16  
- **A100 GPU** (~40GB): embedding batch size = 24

Bu adaptasyon, veri hazırlama sırasında yoğun embedding işlemleri sırasında bellek aşımını (OOM) önler. Aynı zamanda, dataset hazırlama işini paralelleştirmek yerine sırasıyla yaparak GPU üzerindeki basıncı azaltır.

Eğer yine de bellek sorunu yaşarsanız, notebook parametrelerinde `EMBEDDING_BATCH_SIZE` değerini manuel olarak daha düşük bir değere (örn. 4) ayarlayabilirsiniz.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from urllib.parse import urlsplit, urlunsplit

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')


def _resolve_github_token():
    for env_name in ('GH_TOKEN', 'GITHUB_TOKEN'):
        token = str(os.environ.get(env_name, '')).strip()
        if token:
            os.environ.setdefault('GH_TOKEN', token)
            return token

    try:
        import google.colab  # noqa: F401
    except Exception:
        return None

    try:
        from google.colab import userdata
    except Exception:
        return None

    for secret_name in ('GH_TOKEN', 'GITHUB_TOKEN'):
        token = str(userdata.get(secret_name) or '').strip()
        if token:
            os.environ['GH_TOKEN'] = token
            return token

    return None


def _build_repo_access_url(repo_url: str, token: str | None):
    cleaned_url = str(repo_url or '').strip()
    if not cleaned_url or not token:
        return cleaned_url

    parsed = urlsplit(cleaned_url)
    if parsed.scheme != 'https' or not parsed.netloc:
        return cleaned_url

    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, f'x-access-token:{token}@{netloc}', parsed.path, parsed.query, parsed.fragment))


def _ensure_aads_repo_on_path():
    candidates = [Path.cwd(), Path.cwd().parent, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root

    if not CLONE_TARGET.exists():
        CLONE_TARGET.parent.mkdir(parents=True, exist_ok=True)
        clone_url = _build_repo_access_url(REPO_URL, _resolve_github_token())
        completed = subprocess.run(
            ['git', 'clone', '--depth', '1', clone_url, str(CLONE_TARGET)],
            check=False,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        if completed.stdout:
            print(completed.stdout)
        if completed.returncode != 0:
            raise RuntimeError(
                'Notebook 0 repo bootstrap failed. Set AADS_REPO_ROOT to an existing checkout '\
                'or provide GH_TOKEN/GITHUB_TOKEN for private GitHub auto-clone.'
            )

    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET

_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb0_cell02_bootstrap.py', globals())

In [ ]:
with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    # --- Notebook 0 parametreleri ---
    # Once bu hucreyi duzenleyin, sonra alttaki hucreleri sirayla calistirin.

    # --- HIZLI KULLANIM: Genelde sadece bu satirlari degistirin ---
    # REPO_DATASET_ROOT: repo ici relatif yol veya mutlak yol olabilir.
    REPO_DATASET_ROOT = "data/class_root_dataset"

    # REPO_DATASET_NAME: parent klasor verdiyseniz dataset adi; dogrudan dataset yolu verdiyseniz bos birakin.
    REPO_DATASET_NAME = ""

    # DATASET_ROOT: opsiyonel override. Bos ise yukaridaki secim kullanilir.
    DATASET_ROOT = ""

    # IMPORT_FROM_DRIVE: True ise Drive path doluyken Drive seceneklerini sorar; varsayilan akis repo-icidir.
    IMPORT_FROM_DRIVE = False

    # DRIVE_DATASET_PATH: Opsiyonel Drive dataset parent yolu. Bos ise repo data klasoru kullanilir.
    DRIVE_DATASET_PATH = ""

    # DRIVE_DATASET_NAME: Kopyalanacak klasor adi. Bos ise secenekleri listeler ve sorar.
    DRIVE_DATASET_NAME = ""

    # CROP_NAME / PART_NAME: bos birakirsan notebook sorar.
    CROP_NAME = ""
    PART_NAME = ""

    # AUTORUN_AUDIT: True ise audit hucreleri varsayilan akista calisir.
    AUTORUN_AUDIT = True

    # AUTORUN_MATERIALIZE: True ise runtime dataset materyalizasyonu etkin olur.
    AUTORUN_MATERIALIZE = False

    # EMBEDDING_BATCH_SIZE: embedding hesaplamasi sirasinda batch boyutu.
    # Eger bellek sorunu yasarsan bu degeri manuel olarak azaltabilirsin (ornegin 4 veya 6).
    # Varsayilan olarak GPU bellegine gore otomatik ayarlanir:
    # T4 (~16GB): 8, V100 (~32GB): 16, A100 (~40GB): 24
    # EMBEDDING_BATCH_SIZE = 8  # Manuel override icin bu satiri uncomment et

    # MAX_IMAGES_IN_MEMORY: Bellek-akiskan islem icin ayni anda bellekte tutulacak max resim sayisi.
    # Otomatik olarak sistem RAM'e gore ayarlanir:
    # - Yuksek RAM (>=15GB): 256 resim
    # - Dusuk RAM (<15GB): 64 resim
    # Eger hala OOM yasarsan bunu azaltabilirsin (ornegin 32).
    # MAX_IMAGES_IN_MEMORY = 32  # Manuel override icin bu satiri uncomment et

    run_id = RUN_ID
    STATE = {
        "validated": False,
        "audit_summary": None,
        "artifact_root": None,
        "runtime_dataset_root": None,
        "dataset_root": None,
        "dataset_name": None,
        "dataset_source": None,
        "selected_ood_dataset_name": None,
        "resolved_ood_root": None,
        "prep_materialization_result": None,
        "access_report": None,
        "runtime_dataset_push_report": None,
    }
    print(
        f"[PARAMS] run_id={run_id} crop={CROP_NAME or '(ask)'} part={PART_NAME or '(ask)'} "
        f"repo_dataset_root={REPO_DATASET_ROOT} repo_dataset_name={REPO_DATASET_NAME or '(ask)'} "
        f"dataset_root_override={DATASET_ROOT or '(none)'}"
    )
    print(
        f"[DRIVE] import_from_drive={IMPORT_FROM_DRIVE} drive_path={DRIVE_DATASET_PATH or '<none>'} "

In [ ]:
with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    # --- Notebook 0 parametreleri ---
    # Once bu hucreyi duzenleyin, sonra alttaki hucreleri sirayla calistirin.

    # --- HIZLI KULLANIM: Genelde sadece bu satirlari degistirin ---
    # REPO_DATASET_ROOT: repo ici relatif yol veya mutlak yol olabilir.
    REPO_DATASET_ROOT = "data/class_root_dataset"

    # REPO_DATASET_NAME: parent klasor verdiyseniz dataset adi; dogrudan dataset yolu verdiyseniz bos birakin.
    REPO_DATASET_NAME = ""

    # DATASET_ROOT: opsiyonel override. Bos ise yukaridaki secim kullanilir.
    DATASET_ROOT = ""

    # IMPORT_FROM_DRIVE: True ise Drive path doluyken Drive seceneklerini sorar; varsayilan akis repo-icidir.
    IMPORT_FROM_DRIVE = False

    # DRIVE_DATASET_PATH: Opsiyonel Drive dataset parent yolu. Bos ise repo data klasoru kullanilir.
    DRIVE_DATASET_PATH = ""

    # DRIVE_DATASET_NAME: Kopyalanacak klasor adi. Bos ise secenekleri listeler ve sorar.
    DRIVE_DATASET_NAME = ""

    # CROP_NAME / PART_NAME: bos birakirsan notebook sorar.
    CROP_NAME = ""
    PART_NAME = ""

    # AUTORUN_AUDIT: True ise audit hucreleri varsayilan akista calisir.
    AUTORUN_AUDIT = True

    # AUTORUN_MATERIALIZE: True ise runtime dataset materyalizasyonu etkin olur.
    AUTORUN_MATERIALIZE = False

    # EMBEDDING_BATCH_SIZE: embedding hesaplamasi sirasinda batch boyutu.
    # Eger bellek sorunu yasarsan bu degeri manuel olarak azaltabilirsin (ornegin 4 veya 6).
    # Varsayilan olarak GPU bellegine gore otomatik ayarlanir:
    # T4 (~16GB): 8, V100 (~32GB): 16, A100 (~40GB): 24
    # EMBEDDING_BATCH_SIZE = 8  # Manuel override icin bu satiri uncomment et

    run_id = RUN_ID
    STATE = {
        "validated": False,
        "audit_summary": None,
        "artifact_root": None,
        "runtime_dataset_root": None,
        "dataset_root": None,
        "dataset_name": None,
        "dataset_source": None,
        "selected_ood_dataset_name": None,
        "resolved_ood_root": None,
        "prep_materialization_result": None,
        "access_report": None,
        "runtime_dataset_push_report": None,
    }
    print(
        f"[PARAMS] run_id={run_id} crop={CROP_NAME or '(ask)'} part={PART_NAME or '(ask)'} "
        f"repo_dataset_root={REPO_DATASET_ROOT} repo_dataset_name={REPO_DATASET_NAME or '(ask)'} "
        f"dataset_root_override={DATASET_ROOT or '(none)'}"
    )
    print(
        f"[DRIVE] import_from_drive={IMPORT_FROM_DRIVE} drive_path={DRIVE_DATASET_PATH or '<none>'} "

In [ ]:
with TELEMETRY.capture_cell_output("Cell 3a: Google Drive Dataset Import (Opsiyonel)"):
    from scripts.colab_dataset_layout import resolve_dataset_directory_from_parent
    from scripts.colab_repo_bootstrap import mount_drive_if_available
    import shutil
    
    def import_dataset_from_drive(
        source_path: str | Path,
        destination_path: str | Path,
        dataset_name: str,
        overwrite: bool = False
    ) -> bool:
        """Drive'dan dataseti yerel klasore kopyala."""
        source_path = Path(source_path)
        destination_path = Path(destination_path).expanduser()
        destination_path.mkdir(parents=True, exist_ok=True)
        
        target = destination_path / dataset_name
        
        if target.exists():
            if not overwrite:
                print(f"[DRIVE] Hedef klasor zaten var: {target}")
                return False
            shutil.rmtree(target)
            print(f"[DRIVE] Mevcut klasor silindi: {target}")
        
        try:
            print(f"[DRIVE] Kopyalama baslatildi: {source_path} -> {target}")
            shutil.copytree(source_path, target, dirs_exist_ok=False)
            print(f"[DRIVE] Basariyla kopyalandi: {target}")
            return True
        except Exception as e:
            print(f"[DRIVE] Kopyalama basarisiz: {e}")
            return False
    
    def _drive_destination_parent() -> Path:
        explicit_root = str(DATASET_ROOT).strip()
        if explicit_root:
            root_path = Path(explicit_root).expanduser()
            return root_path.resolve() if root_path.is_absolute() else (ROOT / root_path).resolve()
        return ROOT / "data" / "imported_from_drive"
    
    # Google Drive'dan dataset alma secenegi (parametrelerden al)
    if IMPORT_FROM_DRIVE and str(DRIVE_DATASET_PATH).strip():
        print("[DRIVE] Google Drive icinde dataset ara...")
        mount_drive_if_available()
        
        drive_path = Path(DRIVE_DATASET_PATH).expanduser()
        if not drive_path.exists():
            raise RuntimeError(f"Drive yolu bulunamadi: {drive_path}")
        dataset_name, source, available_datasets = resolve_dataset_directory_from_parent(
            dataset_parent=drive_path,
            requested_name=DRIVE_DATASET_NAME,
            prompt_label="Drive dataset",
        )
        print(f"[DRIVE] secilebilir datasetler={available_datasets}")
        dest_parent = _drive_destination_parent()
        if import_dataset_from_drive(source, dest_parent, dataset_name):
            DATASET_ROOT = str(dest_parent / dataset_name)
            print(f"[DRIVE] DATASET_ROOT guncellendi: {DATASET_ROOT}")
        else:
            existing_target = dest_parent / dataset_name
            if existing_target.is_dir():
                DATASET_ROOT = str(existing_target)
                print(f"[DRIVE] Mevcut kopya kullaniliyor, DATASET_ROOT guncellendi: {DATASET_ROOT}")
    else:
        print("[DRIVE] IMPORT_FROM_DRIVE=False veya DRIVE_DATASET_PATH bos. Drive import atlanacak.")

In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb0_cell06_access_check.py', globals())


In [ ]:
with TELEMETRY.capture_cell_output("Cell 4: Dataset Audit"):
    from scripts.colab_dataset_layout import resolve_repo_dataset_directory
    from scripts.evaluate_dataset_layout import evaluate_layout
    from scripts.prepare_grouped_runtime_dataset import build_grouped_dataset_plan, build_human_review_packet, format_human_review_packet
    from src.shared.json_utils import write_json

    def _resolve_repo_dataset_root(repo_relative_root: str) -> Path:
        raw_repo_relative_root = str(repo_relative_root or "").strip()
        if not raw_repo_relative_root:
            raise RuntimeError("REPO_DATASET_ROOT bos birakilamaz.")
        candidate = Path(raw_repo_relative_root).expanduser()
        if candidate.is_absolute():
            resolved = candidate.resolve()
        else:
            resolved = (ROOT / candidate).resolve()
            try:
                resolved.relative_to(ROOT)
            except ValueError as exc:
                raise RuntimeError(f"REPO_DATASET_ROOT repo disina cikamaz: {raw_repo_relative_root}") from exc
        return resolved

    def _normalize_token(value: str) -> str:
        normalized = "".join(ch.lower() if ch.isalnum() else "_" for ch in str(value or "").strip())
        while "__" in normalized:
            normalized = normalized.replace("__", "_")
        return normalized.strip("_")

    def _infer_crop_and_part_from_dataset_name(dataset_name: str) -> tuple[str, str]:
        tokens = [token for token in _normalize_token(dataset_name).split("_") if token]
        if not tokens:
            return "", "unspecified"
        if len(tokens) == 1:
            return tokens[0], "unspecified"
        return tokens[0], "_".join(tokens[1:]) or "unspecified"

    def _prompt_text(prompt: str, default_value: str = "") -> str:
        try:
            raw = str(input(prompt)).strip()
        except EOFError:
            raw = ""
        return raw or str(default_value or "").strip()

    def _prompt_yes_no(prompt: str, default_value: bool = True) -> bool:
        suffix = "[Y/n]" if default_value else "[y/N]"
        try:
            raw = str(input(f"{prompt} {suffix} ")).strip().lower()
        except EOFError:
            raw = ""
        if not raw:
            return bool(default_value)
        return raw in {"y", "yes", "evet", "e"}

    explicit_dataset_root = str(DATASET_ROOT).strip()
    if explicit_dataset_root:
        dataset_root = _resolve_repo_dataset_root(explicit_dataset_root)
        dataset_name = dataset_root.name
    else:
        dataset_name, dataset_root, available_dataset_names = resolve_repo_dataset_directory(
            repo_root=ROOT,
            repo_relative_root=REPO_DATASET_ROOT,
            requested_name=REPO_DATASET_NAME,
            prompt_label="class-root dataset",
        )
        print(f"[PREP] repo dataset options={available_dataset_names}")
    dataset_source = "drive" if IMPORT_FROM_DRIVE and str(DRIVE_DATASET_PATH).strip() and str(DATASET_ROOT).strip() else "repo"
    if not dataset_root.is_dir():
        raise RuntimeError(f"Dataset root bulunamadi: {dataset_root}")
    layout_report = evaluate_layout(root=dataset_root)
    print(
        f"[DATASET] layout_ok={layout_report.get('ok')} "
        f"classes={len(layout_report.get('classes', []))} "
        f"warnings={len(layout_report.get('warnings', []))} "
        f"errors={len(layout_report.get('errors', []))}"
    )
    if not layout_report.get("ok"):
        raise RuntimeError("Dataset layout hatali: " + "; ".join(layout_report.get("errors", [])))

    inferred_crop_name, inferred_part_name = _infer_crop_and_part_from_dataset_name(dataset_name)
    resolved_crop_name = str(CROP_NAME).strip()
    if not resolved_crop_name:
        resolved_crop_name = _prompt_text(
            f"CROP_NAME bos. '{dataset_name}' dataseti icin crop anahtarini girin [{inferred_crop_name or 'crop'}]: ",
            inferred_crop_name or "crop",
        )
    resolved_part_name = str(PART_NAME).strip()
    if not resolved_part_name:
        resolved_part_name = _prompt_text(
            f"PART_NAME bos. '{dataset_name}' dataseti icin part adini girin [{inferred_part_name or 'unspecified'}]: ",
            inferred_part_name or "unspecified",
        ) or "unspecified"
    CROP_NAME = resolved_crop_name
    PART_NAME = resolved_part_name
    print(f"[PREP] secilen crop={CROP_NAME} part={PART_NAME}")

    artifact_root = Path(PREP_ARTIFACT_ROOT).expanduser()
    if not artifact_root.is_absolute():
        artifact_root = (ROOT / artifact_root).resolve()

    summary = build_grouped_dataset_plan(
        class_root=dataset_root,
        crop_name=CROP_NAME,
        artifact_root=artifact_root,
        taxonomy_path=ROOT / "config" / "plant_taxonomy.json",
        dino_model_id=PREP_DINOV3_MODEL_ID,
        bioclip_model_id=PREP_BIOCLIP_MODEL_ID,
        device=DEVICE,
        batch_size=EMBEDDING_BATCH_SIZE,
        neighbors=NEIGHBORS,
        under_min_eval_policy=UNDER_MIN_EVAL_POLICY,
        progress_fn=lambda message: print(f"[PREP] {message}"),
    )

    STATE["validated"] = True
    STATE["audit_summary"] = summary
    STATE["artifact_root"] = artifact_root
    STATE["dataset_root"] = dataset_root
    STATE["dataset_name"] = dataset_name
    STATE["dataset_source"] = dataset_source
    STATE["crop_name"] = CROP_NAME
    STATE["part_name"] = PART_NAME
    print(f"[PREP] dataset_source={dataset_source} dataset_name={dataset_name} dataset_root={dataset_root}")
    print(f"[PREP] crop_name={CROP_NAME} part_name={PART_NAME}")
    print(json.dumps(summary.get("summary", {}), indent=2))
    print(f"[PREP] runtime_ready={summary.get('runtime_ready')} artifact_root={artifact_root}")
    review_packet = build_human_review_packet(
        summary,
        artifact_root=artifact_root,
        max_review_items=MAX_INTERACTIVE_REVIEW_ITEMS,
    )
    STATE["human_review_packet"] = review_packet
    STATE["human_review_approved"] = not bool(review_packet.get("pause_recommended"))
    write_json(artifact_root / "human_review_packet.json", review_packet, ensure_ascii=False)
    print(format_human_review_packet(review_packet))
    if summary.get("blocking_issues"):
        print("[PREP] Bloklayici sorunlar:")
        for item in summary["blocking_issues"]:
            print(f"  - {item}")
    if INTERACTIVE_AUDIT_REVIEW and review_packet.get("pause_recommended"):
        print("[PREP] Human review gate: artifactleri inceleyebilirsiniz; Enter guvenli varsayilani uygular.")
        if not _prompt_yes_no("Audit gate kararini kabul edip guvenli varsayilan routing ile devam edilsin mi?", True):
            STATE["human_review_stop_requested"] = True
            STATE["human_review_approved"] = False
            raise RuntimeError("Human review tarafindan durduruldu. Artifactleri inceleyip hucreyi yeniden calistirin.")
        STATE["human_review_stop_requested"] = False
        STATE["human_review_approved"] = True
    else:
        STATE["human_review_stop_requested"] = False
        STATE["human_review_approved"] = True
    TELEMETRY.update_latest(
        {
            "phase": "data_prep_audited",
            "dataset_root": str(dataset_root),
            "dataset_name": str(dataset_name),
            "dataset_source": dataset_source,
            "artifact_root": str(artifact_root),
            "runtime_ready": bool(summary.get("runtime_ready")),
        }
    )


In [ ]:
with TELEMETRY.capture_cell_output("Cell 4b: Prepare Dataset For Materialization"):
    from scripts.prepare_grouped_runtime_dataset import build_human_review_packet, build_prepared_dataset_key, format_human_review_packet
    from scripts.prepare_materialization_dataset import prepare_class_root_for_materialization
    from src.shared.json_utils import write_json

    if not STATE.get("validated") or STATE.get("audit_summary") is None:
        raise RuntimeError("Once dataset audit hucresini calistirin.")

    summary = STATE["audit_summary"]
    if not PREPARE_DATASET_FROM_REPORTS:
        print("[PREP] PREPARE_DATASET_FROM_REPORTS=False. Audit raporlari pasif birakildi.")
    else:
        dataset_key = build_prepared_dataset_key(CROP_NAME, PART_NAME)
        prepared_class_root_parent = Path(PREPARED_CLASS_ROOT).expanduser()
        if not prepared_class_root_parent.is_absolute():
            prepared_class_root_parent = (ROOT / prepared_class_root_parent).resolve()
        prepared_class_root = prepared_class_root_parent / dataset_key
        prepared_artifact_root = STATE["artifact_root"].parent / f"{STATE['artifact_root'].name}_prepared"
        prep_counts = dict(summary.get("summary", {}))
        print("[PREP] Rapor tabanli hazirlik ozeti:")
        print(
            f"  dataset_key={dataset_key} total_images={prep_counts.get('total_images', 0)} "
            f"cross_class_conflicts={prep_counts.get('cross_class_conflicts', 0)} "
            f"same_class_high_risk_clusters={prep_counts.get('same_class_high_risk_clusters', 0)}"
        )
        print(f"  source_dataset_root={STATE['dataset_root']}")
        print(f"  source_artifact_root={STATE['artifact_root']}")
        print(f"  prepared_class_root={prepared_class_root}")
        print(f"  prepared_artifact_root={prepared_artifact_root}")
        print(f"  cleanup_seed={CLEANUP_SEED}")
        prep_result = prepare_class_root_for_materialization(
            class_root=STATE["dataset_root"],
            crop_name=CROP_NAME,
            part_name=PART_NAME,
            audit_artifact_root=STATE["artifact_root"],
            prepared_class_root=prepared_class_root,
            prepared_artifact_root=prepared_artifact_root,
            taxonomy_path=ROOT / "config" / "plant_taxonomy.json",
            dino_model_id=PREP_DINOV3_MODEL_ID,
            bioclip_model_id=PREP_BIOCLIP_MODEL_ID,
            device=DEVICE,
            batch_size=EMBEDDING_BATCH_SIZE,
            neighbors=NEIGHBORS,
            cleanup_seed=CLEANUP_SEED,
            quarantine_cross_class_conflicts=True,
            under_min_eval_policy=UNDER_MIN_EVAL_POLICY,
            materialization_strategy="auto",
            progress_fn=lambda message: print(f"[PREP] {message}"),
        )
        STATE["prep_materialization_result"] = prep_result
        STATE["dataset_root"] = Path(prep_result["prepared_class_root"])
        STATE["dataset_source"] = "prepared_class_root"
        STATE["artifact_root"] = Path(prep_result["prepared_artifact_root"])
        STATE["audit_summary"] = prep_result["rerun_summary"]
        summary = STATE["audit_summary"]
        review_packet = build_human_review_packet(
            summary,
            artifact_root=STATE["artifact_root"],
            max_review_items=MAX_INTERACTIVE_REVIEW_ITEMS,
        )
        STATE["human_review_packet"] = review_packet
        STATE["human_review_approved"] = not bool(review_packet.get("pause_recommended"))
        write_json(STATE["artifact_root"] / "human_review_packet.json", review_packet, ensure_ascii=False)
        print(
            f"[PREP] prepared_runtime_ready={prep_result.get('prepared_runtime_ready')} "
            f"dataset_key={prep_result.get('dataset_key')} prepared_class_root={STATE['dataset_root']}"
        )
        print(f"[PREP] Hazirlik sonrasi artifact_root={STATE['artifact_root']}")
        print(format_human_review_packet(review_packet))
        if INTERACTIVE_AUDIT_REVIEW and review_packet.get("pause_recommended"):
            print("[PREP] Prepared audit gate: Enter guvenli varsayilani uygular.")
            if not _prompt_yes_no("Hazirlanmis dataset ile devam edilsin mi?", True):
                STATE["human_review_stop_requested"] = True
                STATE["human_review_approved"] = False
                raise RuntimeError("Human review tarafindan durduruldu. Artifactleri inceleyip hucreyi yeniden calistirin.")
            STATE["human_review_stop_requested"] = False
            STATE["human_review_approved"] = True
        else:
            STATE["human_review_stop_requested"] = False
            STATE["human_review_approved"] = True
        TELEMETRY.update_latest(
            {
                "phase": "data_prep_prepared",
                "dataset_root": str(STATE["dataset_root"]),
                "dataset_source": str(STATE.get("dataset_source") or "prepared_class_root"),
                "artifact_root": str(STATE["artifact_root"]),
                "runtime_ready": bool(summary.get("runtime_ready")),
            }
        )


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb0_cell09_gitignore_fix.py', globals())


In [ ]:
## GPU ve Sistem RAM Bellek Optimizasyonu (T4 Colab)

Notebook 0, Colab T4 GPU'da kalitiyle çalışması için otomatik bellek adaptasyonunu destekler:

### GPU Bellek Yönetimi
- **T4 GPU** (~16GB): embedding batch size = **8**
- **V100 GPU** (~32GB): embedding batch size = **16**  
- **A100 GPU** (~40GB): embedding batch size = **24**

### Sistem RAM Yönetimi
- **Yüksek bellek** (≥15GB): normal mod, MAX_IMAGES = 256
- **Düşük bellek** (<15GB): ekonomik mod, MAX_IMAGES = 64
- Otomatik algılama: psutil kullanarak toplam ve mevcut RAM ölçülüyor

Bu adaptasyonlar, veri hazırlama sırasında:
- Embedding işlemleri sırasında GPU OOM'ı önler
- Dosya okuma/işleme operasyonlarında RAM basıncını azaltır
- Batch işlemlerini bellek durumuna göre otomatik yeniden boyutlandırır

Eğer yine de bellek sorunu yaşarsanız, `MAX_IMAGES_IN_MEMORY` değerini 32 veya daha düşüğe ayarlayabilirsiniz.